In [1]:
import pandas as pd
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# Load CKD test data

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

PROCESSED_DIR = os.path.join(BASE_DIR, "datasets", "processed", "ckd")

X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "x_test.csv"))
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, "y_test.csv")).squeeze()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (80, 24)
y_test shape: (80,)


In [3]:
# Load trained CKD models

MODEL_DIR = os.path.join(BASE_DIR, "models", "ckd")

models = {
    "Logistic Regression": joblib.load(
        os.path.join(MODEL_DIR, "logistic_regression.pkl")
    ),
    "Decision Tree": joblib.load(
        os.path.join(MODEL_DIR, "decision_tree.pkl")
    ),
    "Random Forest": joblib.load(
        os.path.join(MODEL_DIR, "random_forest.pkl")
    ),
    "SVM": joblib.load(
        os.path.join(MODEL_DIR, "svm.pkl")
    ),
    "XGBoost": joblib.load(
        os.path.join(MODEL_DIR, "xgboost.pkl")
    )
}

print("All CKD models loaded successfully!")

All CKD models loaded successfully!


In [4]:
# Evaluate all CKD models

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)

print(results_df)

                 Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0  Logistic Regression    0.9875        1.0    0.98  0.989899     1.00
1        Decision Tree    0.9875        1.0    0.98  0.989899     0.99
2        Random Forest    1.0000        1.0    1.00  1.000000     1.00
3                  SVM    0.9875        1.0    0.98  0.989899     1.00
4              XGBoost    1.0000        1.0    1.00  1.000000     1.00


In [5]:
# Display CKD model evaluation results as percentages

results_display = results_df.copy()

for column in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]:
    results_display[column] = (
        results_display[column] * 100
    ).round(2)

results_display

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,98.75,100.0,98.0,98.99,100.0
1,Decision Tree,98.75,100.0,98.0,98.99,99.0
2,Random Forest,100.00,100.0,100.0,100.00,100.0
3,SVM,98.75,100.0,98.0,98.99,100.0
4,XGBoost,100.00,100.0,100.0,100.00,100.0


In [6]:
# Confusion Matrix for Random Forest

best_model = models["Random Forest"]

y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[30  0]
 [ 0 50]]


In [7]:
# Classification Report for Random Forest

print("Random Forest Classification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not CKD", "CKD"]
    )
)

Random Forest Classification Report:

              precision    recall  f1-score   support

     Not CKD       1.00      1.00      1.00        30
         CKD       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



In [8]:
# Save CKD evaluation results

EVALUATION_DIR = os.path.join(BASE_DIR, "datasets", "processed", "ckd")

results_display.to_csv(
    os.path.join(EVALUATION_DIR, "model_evaluation.csv"),
    index=False
)

print("CKD evaluation results saved successfully!")

CKD evaluation results saved successfully!


In [9]:
# Select final CKD model

best_model = models["Random Forest"]

FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "best_model.pkl")

joblib.dump(best_model, FINAL_MODEL_PATH)

print("Final CKD model: Random Forest")
print("Saved to:", FINAL_MODEL_PATH)

Final CKD model: Random Forest
Saved to: d:\Explainable-AI-CDSS\models\ckd\best_model.pkl
